In [1]:
# Standard library imports
import argparse
from enum import Enum
import json
import logging
import os
import sys
from textwrap import dedent as ded
from typing import Annotated, Any, Dict, List, NotRequired, Optional, TypedDict
from uuid import UUID
from datetime import datetime
from string import ascii_lowercase, ascii_uppercase

# Third-party imports
from IPython.display import Markdown, display
from langchain_core.callbacks import BaseCallbackHandler
from langchain_core.messages import AIMessage, AnyMessage, BaseMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.outputs import LLMResult
from langchain_core.tools import tool
from langchain_ollama import ChatOllama
from langgraph.graph import END, MessagesState, START, StateGraph, add_messages
from langgraph.prebuilt import ToolNode, tools_condition
import pandas as pd
import pycountry
from pydantic import BaseModel, Field, conlist, constr, model_validator
import requests

# Local application imports
from app_tkns import TMDB_key, TMDB_read_access_token

# Variable assignments
dedent = lambda x: ded(x.strip())

ModuleNotFoundError: No module named 'pycountry'

In [ ]:

class SimpleJSONTraceHandler(BaseCallbackHandler):
    def __init__(self, filepath: str = "simple_traces.json"):
        self.filepath = filepath
        self.traces = []
        
        if os.path.exists(self.filepath):
            try:
                with open(self.filepath, 'r') as f:
                    self.traces = json.load(f)
            except json.JSONDecodeError:
                pass

    def _save_traces(self) -> None:
        with open(self.filepath, 'w') as f:
            json.dump(self.traces, f, indent=4)

    def on_chat_model_start(
        self,
        serialized: Dict[str, Any],
        messages: List[List[BaseMessage]],
        *,
        run_id: UUID,
        **kwargs: Any,
    ) -> None:
        
        # Extract just the role and content from the inputs
        simple_inputs = [{"role": msg.type, "content": str(msg.content)} for msg in messages[0]]

        trace_entry = {
            "run_id": str(run_id),
            "model_info": kwargs.get("invocation_params", {}).get("model", "unknown"),
            "start_time": datetime.utcnow().isoformat() + "Z",
            "status": "running",
            "input_messages": simple_inputs
        }
        
        self.traces.append(trace_entry)
        self._save_traces()

    def on_llm_end(
        self,
        response: LLMResult,
        *,
        run_id: UUID,
        **kwargs: Any,
    ) -> None:
        
        for trace in self.traces:
            if trace.get("run_id") == str(run_id):
                trace["status"] = "completed"
                trace["end_time"] = datetime.utcnow().isoformat() + "Z"
                
                # Extract output and token usage (Ollama specific mapping)
                if response.generations and response.generations[0]:
                    gen = response.generations[0][0]
                    trace["output_messages"] = [{"role": "ai", "content": gen.text}]
                    
                    # Grab Ollama token counts from metadata
                    metadata = getattr(gen, 'message', type('obj', (object,), {'response_metadata': {}})).response_metadata
                    trace["token_usage"] = {
                        "prompt_tokens": metadata.get("prompt_eval_count", 0),
                        "completion_tokens": metadata.get("eval_count", 0)
                    }
                    
                    # Update model_info if Ollama provided a more specific one in the response
                    if metadata.get("model"):
                        trace["model_info"] = metadata.get("model")
                break
                
        self._save_traces()

    def on_llm_error(
        self,
        error: BaseException,
        *,
        run_id: UUID,
        **kwargs: Any,
    ) -> None:
        
        for trace in self.traces:
            if trace.get("run_id") == str(run_id):
                trace["status"] = "failed"
                trace["end_time"] = datetime.utcnow().isoformat() + "Z"
                trace["output_messages"] = [{"role": "error", "content": str(error)}]
                break
                
        self._save_traces()



In [2]:
#llm = ChatOllama(model="llama3.1:8b", temperature=0.0)
#llm = ChatOllama(model="gpt-oss:20b", temperature=0.0, keep_alive=False)
from dotenv import load_dotenv
load_dotenv()

TMDB_API_KEY = os.getenv("TMDB_API_KEY") or os.getenv("TMDB_key")
TMDB_BASE = "https://api.themoviedb.org/3"


In [49]:

# 1. Initialize the custom JSON handler
json_tracer = SimpleJSONTraceHandler(filepath="agent_traces.json")

# 2. Initialize the Ollama Chat Model with the callback
llm = ChatOllama(
    model="gpt-oss:120b-cloud", # Or your preferred local model
    temperature=0,
    callbacks=[json_tracer]
)

In [51]:

# 2. Initialize the Ollama Chat Model with the callback
llm_small = ChatOllama(
    model="llama3.1:8b", # Or your preferred local model
    temperature=0,
    callbacks=[json_tracer]
)

In [29]:

SUBSCRIBED = {"NETFLIX", "AMAZON", "HBO", "MAX", "APPLE", "DISNEY", "HULU", "TUBI"}


def _rq(url: str, params: Dict[str, Any]) -> Dict[str, Any]:
    r = requests.get(url, params=params, timeout=25)
    r.raise_for_status()
    return r.json()


def update_dict(a, b):
    return {**a, **b}


def rename_country(country):
    name = pycountry.countries.get(alpha_2=country).name
    flag = pycountry.countries.get(alpha_2=country).flag

    return name.title()
    return f"{name} {flag}"


def filter_subscribed(x):
    for i in SUBSCRIBED:
        if i in x:
            return True
    return False


ALL_FLAGS = (
    pd.Series(
        {
            f"{a}{b}": pycountry.countries.get(alpha_2=f"{a}{b}")
            for a in ascii_uppercase
            for b in ascii_uppercase
        }
    )
    .dropna()
    .apply(lambda x: f"{x.name} ({x.flag})".upper())
)


In [30]:

def tmdb_table(results) -> pd.DataFrame:
    """Get a table of the info of providers per country"""
    if results:
        sub_df = pd.DataFrame(
            {
                rename_country(country): {
                    k: [i["provider_name"] for i in v]
                    for k, v in results[country].items()
                    if k in ["ads", "flatrate", "free"]
                }
                for country in results.keys()
            }
        )

        if sub_df.empty:
            sub_df = pd.DataFrame(
                {
                    rename_country(country): {
                        k: [i["provider_name"] for i in v]
                        for k, v in results[country].items()
                        if k in ["ads", "rent", "flatrate", "buy", "free"]
                    }
                    for country in results.keys()
                }
            )

        return json.dumps(
            sub_df.stack()
            .explode()
            .sort_index()
            .reset_index()
            .set_axis(["Type", "Country", "Provider"], axis=1)
            .assign(Provider=lambda df: df["Provider"].apply(lambda x: str(x).upper().strip()))
            .loc[lambda df: df["Provider"].apply(filter_subscribed)]
            .pivot_table(index=["Type", "Provider"], values="Country", aggfunc=list)
            .groupby(level=0)
            .apply(lambda g: g.droplevel(0).to_dict()["Country"])
            .to_dict(),
            indent=4,
        )
    return pd.DataFrame()


In [39]:
from typing import Optional, Dict, Any, List
from langchain_core.tools import tool

# Assuming TMDB_API_KEY, TMDB_BASE, _rq, and tmdb_table are defined elsewhere in your file

@tool
def tmdb_search_movie
    title: str, year: Optional[int] = None, director: Optional[str] = None
) -> List[Dict[str, Any]]:
    """
    Search TMDB (The Movie Database) for a movie by its title. 
    Can optionally filter by release year or director.
    Returns a list of up to 10 movie candidates containing their 'id', 'title', 'release_year', and 'overview'.
    """
    if not TMDB_API_KEY:
        return [{"error": "TMDB_API_KEY not set"}]
        
    params = {"api_key": TMDB_API_KEY, "query": title, "include_adult": "false"}
    if year:
        params["year"] = year
        
    data = _rq(f"{TMDB_BASE}/search/movie", params)
    cands = []
    
    for r in (data.get("results") or [])[:10]:
        cands.append(
            {
                "id": r.get("id"),
                "title": r.get("title"),
                "original_language": (r.get("original_language") or ""),
                "release_year": (r.get("release_date") or "")[:4],
                "overview": r.get("overview"),
            }
        )
    return cands


@tool
def tmdb_watch_providers(movie_id: int) -> Dict[str, Any]:
    """
    Get TMDB Watch Providers (where to stream, rent, or buy) categorized by country for a specific movie.
    Requires the numeric movie_id (which can be obtained from tmdb_search_movie).
    """
    if not TMDB_API_KEY:
        return {"error": "TMDB_API_KEY not set"}
        
    data = _rq(f"{TMDB_BASE}/movie/{movie_id}/watch/providers", {"api_key": TMDB_API_KEY})

    results = data.get("results", {})
    if results:
        return tmdb_table(results)

    return {"message": "No watch providers found."}

In [35]:
tmdb_search_movie("Shrek 2")

[{'id': 809,
  'title': 'Shrek 2',
  'original_language': 'en',
  'release_year': '2004',
  'overview': 'Happily ever after never seemed so far far away when a trip to meet the in-laws turns into a hilariously twisted adventure for Shrek and Fiona. With the help of his faithful Donkey, Shrek takes on a potion-brewing Fairy Godmother, the pompous Prince Charming, and the ogre-killer, Puss In Boots.'}]

In [38]:
print(tmdb_watch_providers(809)[:400])

{
    "flatrate": {
        "AMAZON PRIME VIDEO": [
            "Algeria",
            "Andorra",
            "Angola",
            "Antigua And Barbuda",
            "Argentina",
            "Australia",
            "Bahamas",
            "Bahrain",
            "Barbados",
            "Belgium",
            "Belize",
            "Bermuda",
            "Bolivia, Plurinational State Of",
          


In [41]:
# 1. Define the tools array and bind them to the ChatOllama LLM
tools = [tmdb_search_movie, tmdb_watch_providers]
llm_with_tools = llm.bind_tools(tools)

# 2. Define the strict system prompt
system_prompt = SystemMessage(content="""
You are a highly specialized movie assistant. Your capabilities are strictly limited to two actions:
1. Searching for movies to confirm details with the user.
2. Finding streaming platforms where a movie can be watched.

CRITICAL INSTRUCTION:
If a user asks where to stream or watch a movie, you MUST ALWAYS execute the 'tmdb_search_movie' tool FIRST to retrieve the correct TMDB 'id'. 
Under no circumstances should you call 'tmdb_watch_providers' without first obtaining the exact movie ID from the search tool.
""")

# 3. Create the node function for the agent
def agent(state: MessagesState):
    # Ensure the system prompt is always injected at the start of the context window
    messages = [system_prompt] + state["messages"]
    
    # Invoke the LLM (which is aware of the tools)
    response = llm_with_tools.invoke(messages)
    
    # Return the response to be appended to the state
    return {"messages": [response]}


In [42]:

# 4. Build the LangGraph Workflow
workflow = StateGraph(MessagesState)

# Add the standard nodes
workflow.add_node("agent", agent)
workflow.add_node("tools", ToolNode(tools)) # ToolNode automatically executes the requested tools

# Define the flow
workflow.add_edge(START, "agent")

# tools_condition automatically checks if the LLM returned a tool call. 
# If it did, it routes to "tools". If not, it routes to END.
workflow.add_conditional_edges("agent", tools_condition)

# After tools execute, always return the results back to the agent for the final answer
workflow.add_edge("tools", "agent")

# 5. Compile the graph into an executable application
agent_executor = workflow.compile()

In [44]:
resp = agent_executor.invoke({"messages": HumanMessage("Where to stream shrek 2")})

In [48]:
[i.pretty_print() for i in resp["messages"]]

================================ Human Message =================================

Where to stream shrek 2
================================== Ai Message ==================================
Tool Calls:
  tmdb_search_movie (4da7f39b-7046-4526-b8af-988cfabf023a)
 Call ID: 4da7f39b-7046-4526-b8af-988cfabf023a
  Args:
    title: Shrek 2
    year: None
    director: None
================================= Tool Message =================================
Name: tmdb_search_movie

[{"id": 809, "title": "Shrek 2", "original_language": "en", "release_year": "2004", "overview": "Happily ever after never seemed so far far away when a trip to meet the in-laws turns into a hilariously twisted adventure for Shrek and Fiona. With the help of his faithful Donkey, Shrek takes on a potion-brewing Fairy Godmother, the pompous Prince Charming, and the ogre-killer, Puss In Boots."}]
================================== Ai Message ==================================
Tool Calls:
  tmdb_watch_providers (3da79d50-0313-4

[None, None, None, None, None, None]